# State Space Model (SSM) - Mamba
S4 (Gu et al., 2021) -> "Mamba: Linear-Time Sequence Modeling with Selective State Spaces" (Gu & Dao, 2023)

**Idea:** a recurrent model whose recurrence is **linear** in the state: $h_t = \bar{A} h_{t-1} + \bar{B} x_t$, $y_t = C h_t$. No tanh inside the loop means it can be computed with a parallel scan (fast training like a Transformer), while generation keeps a fixed-size state (cheap like an RNN). Mamba makes $\bar A, \bar B, C$ depend on the input (**selective**), so the model can choose what to remember and what to forget.

### SSM vs RNN vs Transformer
| | RNN / LSTM | Transformer (SLM) | SSM (Mamba) |
|---|---|---|---|
| Recurrence | nonlinear: $h_t = \tanh(W h_{t-1} + U x_t)$ | none (attention) | **linear**: $h_t = \bar A_t h_{t-1} + \bar B_t x_t$ |
| Training over time | sequential | parallel, $O(T^2)$ | parallel scan, $O(T)$ |
| Generation cost / token | $O(1)$ state | $O(T)$ (KV cache grows) | $O(1)$ state |
| Memory | fixed $h$ | all past tokens | fixed $h$ ($D_{inner} \times N$ per layer) |
| Gating / selection | LSTM gates | attention weights | input-dependent $\Delta_t, B_t, C_t$ |
| Real examples | - | Llama, GPT | Mamba, Mamba-2, Jamba / Zamba (hybrids with attention) |

Shape flow: `idx (B, T) -> embed (B, T, D) -> [RMSNorm -> in_proj -> conv -> SiLU -> selective scan (B, T, D_in) -> * SiLU(gate) -> out_proj -> +] x L -> logits (B, T, V)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - tiny Shakespeare (character level)
~1.1M characters, vocab = 65 unique characters. Each training example is a random chunk of `block_size` characters; the target is the same chunk shifted by one (predict the next character).

The same data and loss are used in `rnn`, `lstm`, `gru`, `slm`, `moe`, `ssm`, so their val losses can be compared.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))          # (1115394,)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 64   # T: characters per sequence
batch_size = 32   # B

def get_batch(split):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print("vocab size:", vocab_size, "| train chars:", len(train_data), "| val chars:", len(val_data))

## 2. Selective SSM (the core of Mamba)
Each of the $D_{in}$ channels has its own $N$-dim state. $A$ is a fixed learned (negative) diagonal matrix; $\Delta_t$, $B_t$, $C_t$ are computed from the current input.

We run the scan with a plain Python loop over $T$ for readability. Mamba computes the same recurrence with a hardware-aware **parallel** scan.

### Formula: Continuous SSM
$$h'(t) = A\,h(t) + B\,x(t), \qquad y(t) = C\,h(t) + D\,x(t)$$
- $x(t)$: input signal (one channel), $y(t)$: output
- $h(t) \in \mathbb{R}^{N}$: hidden state, $N = 16$ (`d_state`)
- $A \in \mathbb{R}^{N \times N}$: state dynamics, diagonal and negative here ($A = -e^{A_{log}}$), so the state decays and is stable
- $B \in \mathbb{R}^{N \times 1}$, $C \in \mathbb{R}^{1 \times N}$: how input enters the state and how the state is read out
- $D$: skip connection from input to output

### Formula: Discretization (zero-order hold)
$$\bar A_t = \exp(\Delta_t A), \qquad \bar B_t = (\Delta_t A)^{-1}(\exp(\Delta_t A) - I)\,\Delta_t B_t \approx \Delta_t B_t$$
- $\Delta_t > 0$: step size, per channel and per token (`softplus(dt_proj(...))`)
- large $\Delta_t$: $\bar A_t \to 0$, reset the state and focus on the current input
- small $\Delta_t$: $\bar A_t \to 1$, keep the state and mostly ignore this input
- Mamba uses the simplified $\bar B_t \approx \Delta_t B_t$ (as in the code)

### Formula: Selective scan (discrete recurrence)
$$h_t = \bar A_t \odot h_{t-1} + \bar B_t\, x_t, \qquad y_t = C_t\, h_t + D\, x_t$$
- $\odot$: element-wise, since $A$ is diagonal
- **selective**: $\Delta_t = \text{softplus}(W_\Delta x_t)$, $B_t = W_B x_t$, $C_t = W_C x_t$ all depend on $x_t$ (S4 used fixed ones)
- linear in $h$: unrolling gives $h_t = \sum_{s \le t} \left(\prod_{r=s+1}^{t} \bar A_r\right) \bar B_s x_s$, which is what allows a parallel scan

In [ ]:
class SelectiveSSM(nn.Module):
    # (B, T, D_in) -> (B, T, D_in)
    def __init__(self, d_inner, d_state=16, dt_rank=16):
        super().__init__()
        self.d_state, self.dt_rank = d_state, dt_rank
        self.x_proj = nn.Linear(d_inner, dt_rank + 2 * d_state, bias=False)     # x -> (Delta low-rank, B, C)
        self.dt_proj = nn.Linear(dt_rank, d_inner)                              # low-rank Delta -> per channel
        A = torch.arange(1, d_state + 1).float().repeat(d_inner, 1)             # (D_in, N): 1..N (S4D-real init)
        self.A_log = nn.Parameter(torch.log(A))                                 # A = -exp(A_log) < 0
        self.D = nn.Parameter(torch.ones(d_inner))

    def forward(self, x):
        Bsz, T, d_inner = x.shape
        A = -torch.exp(self.A_log)                                              # (D_in, N)
        dt, B, C = self.x_proj(x).split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))                                       # (B, T, D_in), Delta > 0
        dA = torch.exp(dt.unsqueeze(-1) * A)                                    # (B, T, D_in, N) A_bar
        dB = dt.unsqueeze(-1) * B.unsqueeze(2)                                  # (B, T, D_in, N) B_bar

        h = torch.zeros(Bsz, d_inner, self.d_state, device=x.device)           # (B, D_in, N)
        ys = []
        for t in range(T):                                                      # sequential scan
            h = dA[:, t] * h + dB[:, t] * x[:, t, :, None]                      # (B, D_in, N)
            ys.append((h * C[:, t, None, :]).sum(-1))                           # C_t h_t -> (B, D_in)
        y = torch.stack(ys, dim=1)                                              # (B, T, D_in)
        return y + x * self.D

## 3. Mamba block
The SSM sits inside a gated block that replaces **both** attention and the FFN:
1. `in_proj` expands $D \to 2D_{in}$ and splits into a main branch $x$ and a gate $z$
2. a short causal depthwise conv (kernel 4) mixes each channel with its last 3 tokens
3. SiLU, then the selective SSM
4. multiply by $\text{SiLU}(z)$ (a gate, like SwiGLU), project back to $D$

### Formula: Mamba block
$$u = \text{SiLU}(\text{Conv1d}(W_x x)), \qquad \text{out} = W_{out}\left(\text{SSM}(u) \odot \text{SiLU}(W_z x)\right)$$
- $x \in \mathbb{R}^{T \times D}$: block input (after RMSNorm)
- $W_x, W_z \in \mathbb{R}^{D_{in} \times D}$: fused in `in_proj`, $D_{in} = 2D$
- Conv1d: depthwise, left-padded so position $t$ only sees $t-3 \dots t$ (causal)
- $W_{out} \in \mathbb{R}^{D \times D_{in}}$: `out_proj`
- residual and pre-norm are the same as in the SLM: $x \leftarrow x + \text{Mamba}(\text{RMSNorm}(x))$ (RMSNorm formula in `slm.ipynb`)

In [ ]:
class RMSNorm(nn.Module):
    # (..., D) -> (..., D)
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class MambaBlock(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, d_state=16, expand=2, d_conv=4):
        super().__init__()
        d_inner = expand * dim
        self.in_proj = nn.Linear(dim, 2 * d_inner, bias=False)                      # W_x, W_z
        self.conv = nn.Conv1d(d_inner, d_inner, d_conv, groups=d_inner, padding=d_conv - 1)  # depthwise
        self.ssm = SelectiveSSM(d_inner, d_state)
        self.out_proj = nn.Linear(d_inner, dim, bias=False)

    def forward(self, x):
        T = x.shape[1]
        x, z = self.in_proj(x).chunk(2, dim=-1)                     # each (B, T, D_in)
        x = self.conv(x.transpose(1, 2))[..., :T].transpose(1, 2)   # (B, D_in, T+3) -> keep first T = causal
        y = self.ssm(F.silu(x))                                     # (B, T, D_in)
        return self.out_proj(y * F.silu(z))                         # (B, T, D)


class MambaLM(nn.Module):
    # idx (B, T) -> logits (B, T, V)
    def __init__(self, vocab_size, dim=192, depth=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)                  # no positional encoding: the scan is ordered
        self.norms = nn.ModuleList([RMSNorm(dim) for _ in range(depth)])
        self.blocks = nn.ModuleList([MambaBlock(dim) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight                        # weight tying
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, idx, state=None):
        x = self.embed(idx)                                         # (B, T, D)
        for norm, block in zip(self.norms, self.blocks):
            x = x + block(norm(x))                                  # (B, T, D)
        return self.head(self.norm(x)), None                        # (B, T, V)

model = MambaLM(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(model(torch.zeros(2, 10, dtype=torch.long, device=device))[0].shape)

## 4. Training
Same loop and loss as the other LM notebooks. Context is 64 characters (like the RNNs) because our Python-loop scan stores every step for backprop.

### Formula: Next-token cross-entropy
$$\mathcal{L} = -\frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T} \log p_\theta(x_{t+1} \mid x_{\le t})$$
- $B, T$: batch size and sequence length
- $x_{t+1}$: the true next character
- $p_\theta(\cdot \mid x_{\le t})$: softmax over the vocab from the logits at position $t$
- a model that guesses uniformly scores $\log 65 \approx 4.17$; lower is better

### Formula: Gradient clipping (by global norm)
$$g \leftarrow g \cdot \min\!\left(1, \frac{c}{\lVert g \rVert_2}\right)$$
- $g$: all parameter gradients concatenated into one vector
- $\lVert g \rVert_2$: its L2 norm
- $c$: max norm (1.0); rescales the whole gradient, keeps its direction

In [ ]:
max_iters = 3000
eval_every = 500
lr = 2e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

def loss_fn(model, x, y):
    logits, _ = model(x)                                              # (B, T, V)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # (B*T, V) vs (B*T,) -> scalar

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s)).item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    x, y = get_batch("train")
    loss = loss_fn(model, x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 5. Generation
For simplicity we recompute the scan over the recent context for every new character. A real Mamba keeps just the SSM state $h$ (and the last 3 conv inputs) per layer, so each new token costs $O(1)$ like an RNN.

### Formula: Sampling with temperature
$$p_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}, \qquad x_{t+1} \sim \text{Categorical}(p)$$
- $z_i$: logit of character $i$
- $\tau$: temperature (0.8); $\tau < 1$ sharpens (safer), $\tau > 1$ flattens (more random), $\tau \to 0$ = greedy argmax

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)           # (1, t)
    for _ in range(n):
        logits, _ = model(idx[:, -block_size:])                   # re-reads the last block_size chars each step (no KV cache)
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)    # (1, V)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)  # (1, t+1)
    return decode(idx[0].tolist())

print(generate(model))

## Notes
- The pure-PyTorch loop is slow; the `mamba-ssm` package has the fused parallel-scan CUDA kernel.
- Key difference from the LSTM: gating is on a **linear** recurrence, so the whole sequence can be processed in parallel during training, and the state is much larger ($D_{in} \times N$ per layer).
- Pure SSMs are weaker at exact copying/recall from far back (a fixed-size state can't store everything); hybrids like Jamba interleave a few attention layers with Mamba layers.
- **LNN** (next) is another continuous-time recurrent model, but with a *nonlinear*, input-dependent time constant.